# Imports and Paths

In [2]:
# ============================================================
# ADVANCED ANALYTICS
# Mutual Fund Analytics Project
# ============================================================

from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Libraries loaded successfully.")

DATA_PATH = Path("../Data/processed")

print("Current notebook directory:", Path.cwd())
print("Data path:", DATA_PATH.resolve())

print("\nFiles available:")

if DATA_PATH.exists():
    for file in sorted(DATA_PATH.glob("*.csv")):
        print(" -", file.name)
else:
    print("ERROR: Data path does not exist.")

Libraries loaded successfully.
Current notebook directory: c:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\notebooks
Data path: C:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\Data\Processed

Files available:
 - 01_fund_metadata.csv
 - 02_nav_history.csv
 - 03_aum_by_fund_house.csv
 - 04_monthly_sip_inflows.csv
 - 05_category_inflows.csv
 - 06_industry_folio_count.csv
 - 07_scheme_performance.csv
 - 08_investor_transactions.csv
 - 09_portfolio_holdings.csv
 - 10_benchmark_indices.csv


## Loading Datasets

In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# DATA PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "Data" / "processed"

print("Data path:")
print(DATA_PATH)

print("\nFiles available:")
for file in sorted(DATA_PATH.glob("*.csv")):
    print(file.name)

Data path:
c:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\Data\processed

Files available:
01_fund_metadata.csv
02_nav_history.csv
03_aum_by_fund_house.csv
04_monthly_sip_inflows.csv
05_category_inflows.csv
06_industry_folio_count.csv
07_scheme_performance.csv
08_investor_transactions.csv
09_portfolio_holdings.csv
10_benchmark_indices.csv


In [4]:
# ============================================================
# LOAD DATASETS
# ============================================================

fund_metadata = pd.read_csv(
    DATA_PATH / "01_fund_metadata.csv"
)

nav_history = pd.read_csv(
    DATA_PATH / "02_nav_history.csv"
)

aum = pd.read_csv(
    DATA_PATH / "03_aum_by_fund_house.csv"
)

sip = pd.read_csv(
    DATA_PATH / "04_monthly_sip_inflows.csv"
)

category = pd.read_csv(
    DATA_PATH / "05_category_inflows.csv"
)

folio = pd.read_csv(
    DATA_PATH / "06_industry_folio_count.csv"
)

scheme = pd.read_csv(
    DATA_PATH / "07_scheme_performance.csv"
)

investor = pd.read_csv(
    DATA_PATH / "08_investor_transactions.csv"
)

portfolio = pd.read_csv(
    DATA_PATH / "09_portfolio_holdings.csv"
)

benchmark = pd.read_csv(
    DATA_PATH / "10_benchmark_indices.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
#validation cell.
datasets = {
    "fund_metadata": fund_metadata,
    "nav_history": nav_history,
    "aum": aum,
    "sip": sip,
    "category": category,
    "folio": folio,
    "scheme": scheme,
    "investor": investor,
    "portfolio": portfolio,
    "benchmark": benchmark
}

for name, df in datasets.items():
    print(f"{name:20} {df.shape}")

fund_metadata        (35, 7)
nav_history          (97828, 3)
aum                  (90, 5)
sip                  (48, 6)
category             (144, 3)
folio                (21, 6)
scheme               (40, 19)
investor             (32778, 13)
portfolio            (322, 8)
benchmark            (8050, 3)


In [6]:
# Basic Standardization
nav_history.columns = (
    nav_history.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

fund_metadata.columns = (
    fund_metadata.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

investor.columns = (
    investor.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

portfolio.columns = (
    portfolio.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

scheme.columns = (
    scheme.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Columns standardized.")

Columns standardized.


## Preparing NAV returns

In [7]:
# DAILY RETURNS
nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    errors="coerce"
)

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history["scheme_code"] = pd.to_numeric(
    nav_history["scheme_code"],
    errors="coerce"
)

nav_history = nav_history.dropna(
    subset=[
        "date",
        "nav",
        "scheme_code"
    ]
)

nav_history = nav_history[
    nav_history["nav"] > 0
].copy()

nav_history = nav_history.sort_values(
    ["scheme_code", "date"]
)

nav_history["daily_return"] = (
    nav_history
    .groupby("scheme_code")["nav"]
    .pct_change()
)

returns = nav_history.dropna(
    subset=["daily_return"]
).copy()

print(
    "Valid schemes:",
    returns["scheme_code"].nunique()
)

print(
    "Return observations:",
    len(returns)
)

Valid schemes: 34
Return observations: 97794


## Historical VaR/CVaR

In [8]:
var_results = []

for scheme_code, group in returns.groupby("scheme_code"):

    daily_returns = group["daily_return"].dropna()

    var_95 = daily_returns.quantile(0.05)

    tail_returns = daily_returns[
        daily_returns <= var_95
    ]

    cvar_95 = tail_returns.mean()

    scheme_name = (
        group["scheme_name"].iloc[0]
        if "scheme_name" in group.columns
        else str(scheme_code)
    )

    var_results.append({
        "scheme_code": scheme_code,
        "scheme_name": scheme_name,
        "observations": len(daily_returns),
        "VaR_95": var_95,
        "CVaR_95": cvar_95
    })

var_cvar = pd.DataFrame(var_results)

var_cvar = var_cvar.sort_values(
    "VaR_95"
).reset_index(drop=True)

display(var_cvar)

,scheme_code,scheme_name,observations,VaR_95,CVaR_95
0,100033,100033,5005,-0.0201,-0.0321
1,119599,119599,872,-0.0185,-0.0260
2,120843,120843,3345,-0.0178,-0.0270
3,120842,120842,3345,-0.0178,-0.0270
4,149323,149323,1145,-0.0166,-0.0246
5,149324,149324,1145,-0.0166,-0.0239
6,149322,149322,1145,-0.0166,-0.0244
7,118634,118634,3341,-0.0164,-0.0274
8,119095,119095,3344,-0.0159,-0.0262
9,118633,118633,3341,-0.0159,-0.0246


In [9]:
# ATTACH FUND METADATA
metadata_lookup = fund_metadata[
    [
        "scheme_code",
        "scheme_name",
        "fund_house"
    ]
].copy()

metadata_lookup["scheme_code"] = pd.to_numeric(
    metadata_lookup["scheme_code"],
    errors="coerce"
)

metadata_lookup = metadata_lookup.drop_duplicates(
    subset=["scheme_code"]
)

if "scheme_name" not in var_cvar.columns:
    
    var_cvar = var_cvar.merge(
        metadata_lookup,
        on="scheme_code",
        how="left",
        validate="one_to_one"
    )

else:
    
    var_cvar = var_cvar.merge(
        metadata_lookup[
            [
                "scheme_code",
                "fund_house"
            ]
        ],
        on="scheme_code",
        how="left",
        validate="one_to_one"
    )

display(var_cvar)

,scheme_code,scheme_name,observations,VaR_95,CVaR_95,fund_house
0,100033,100033,5005,-0.0201,-0.0321,Aditya Birla Sun Life Mutual Fund
1,119599,119599,872,-0.0185,-0.0260,Sundaram Mutual Fund
2,120843,120843,3345,-0.0178,-0.0270,quant Mutual Fund
3,120842,120842,3345,-0.0178,-0.0270,quant Mutual Fund
4,149323,149323,1145,-0.0166,-0.0246,ITI Mutual Fund
5,149324,149324,1145,-0.0166,-0.0239,ITI Mutual Fund
6,149322,149322,1145,-0.0166,-0.0244,ITI Mutual Fund
7,118634,118634,3341,-0.0164,-0.0274,Nippon India Mutual Fund
8,119095,119095,3344,-0.0159,-0.0262,DSP Mutual Fund
9,118633,118633,3341,-0.0159,-0.0246,Nippon India Mutual Fund


In [10]:
# EXPORT VaR / CVaR REPORT
REPORT_PATH = Path("../reports")

REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

var_cvar.to_csv(
    REPORT_PATH / "var_cvar_report.csv",
    index=False
)

print(
    "Saved:",
    REPORT_PATH / "var_cvar_report.csv"
)

Saved: ..\reports\var_cvar_report.csv


## Rolling Sharpe

In [11]:
# ROLLING 90-DAY SHARPE
rolling_data = returns.copy()

rolling_data = rolling_data.sort_values(
    ["scheme_code", "date"]
)

rolling_data["rolling_mean_90"] = (
    rolling_data
    .groupby("scheme_code")["daily_return"]
    .transform(
        lambda x: x.rolling(90).mean()
    )
)

rolling_data["rolling_std_90"] = (
    rolling_data
    .groupby("scheme_code")["daily_return"]
    .transform(
        lambda x: x.rolling(90).std()
    )
)

rolling_data["rolling_sharpe_90"] = (
    rolling_data["rolling_mean_90"]
    / rolling_data["rolling_std_90"]
) * np.sqrt(252)

print(
    "Rolling Sharpe calculated."
)

print(
    "Valid rolling observations:",
    rolling_data["rolling_sharpe_90"].notna().sum()
)

Rolling Sharpe calculated.
Valid rolling observations: 93338


In [12]:
# VERIFY ROLLING SHARPE
print(
    "Schemes:",
    rolling_data["scheme_code"].nunique()
)

display(
    rolling_data[
        [
            "date",
            "scheme_code",
            "daily_return",
            "rolling_sharpe_90"
        ]
    ]
    .dropna(subset=["rolling_sharpe_90"])
    .head(20)
)

Schemes: 34


,date,scheme_code,daily_return,rolling_sharpe_90
90,2006-08-11,100033,0.0044,-0.9656
91,2006-08-14,100033,0.0092,-0.9142
92,2006-08-16,100033,0.0115,-0.8630
93,2006-08-17,100033,-0.0001,-0.7567
94,2006-08-18,100033,0.0034,-0.7812
95,2006-08-21,100033,0.0053,-0.5109
96,2006-08-22,100033,-0.0043,-0.3748
97,2006-08-23,100033,-0.0066,-0.5763
98,2006-08-24,100033,0.0084,-0.6522
99,2006-08-25,100033,0.0043,-0.6540


In [13]:
investor.columns.tolist()

['investor_id',
 'transaction_date',
 'amfi_code',
 'transaction_type',
 'amount_inr',
 'state',
 'city',
 'city_tier',
 'age_group',
 'gender',
 'annual_income_lakh',
 'payment_mode',
 'kyc_status']

In [14]:
# Preparing investor cohort data
investor_cohort = investor.copy()

# Ensure transaction date is datetime
investor_cohort["transaction_date"] = pd.to_datetime(
    investor_cohort["transaction_date"],
    errors="coerce"
)

# Ensure amount is numeric
investor_cohort["amount_inr"] = pd.to_numeric(
    investor_cohort["amount_inr"],
    errors="coerce"
)

# Remove invalid records
investor_cohort = investor_cohort.dropna(
    subset=["investor_id", "transaction_date", "amount_inr"]
)

# Sort transactions chronologically
investor_cohort = investor_cohort.sort_values(
    ["investor_id", "transaction_date"]
)

print("Valid investor transactions:", len(investor_cohort))
print(
    "Date range:",
    investor_cohort["transaction_date"].min(),
    "to",
    investor_cohort["transaction_date"].max()
)

Valid investor transactions: 13048
Date range: 2024-01-01 00:00:00 to 2025-12-05 00:00:00
